# s14 - Evasion attack: evaluasi R5 (SURVEY)

**Klaim R5** (`paper-survey.md`): NIDS berbasis ML rapuh terhadap evasion
[F28][F33][F34]. Kami serang model global FL terlatih (dari R3) dengan serangan
yang **sudah ada** di literatur, pada **ruang fitur ter-standardisasi (z-space)**,
dengan perturbasi di-**clip** ke persentil 1-99 tiap fitur. Tidak ada aturan konsistensi
protokol antar-fitur (mis. byte vs paket): hasil = taksiran kerentanan di ruang fitur.

**Serangan diuji:** FGSM, PGD (iteratif), JSMA/saliency (pilih fitur paling
berpengaruh). Semua white-box pada MLP.

**PROTOKOL (dikoreksi).** Model dilatih di EC2 dengan z-score LOKAL per klien, maka tiap
test set klien dinormalisasi dengan statistik train klien itu, dan batas clip diambil dari
train klien itu juga. Hasil dilaporkan per klien (`cic`, `unsw`) dan `union`.

**Definisi ASR (dikoreksi).** `asr` = proporsi serangan yang **terdeteksi sebelum diserang**
tetapi lolos sesudah perturbasi. `raw_miss` = proporsi seluruh serangan yang diprediksi
benign sesudah perturbasi (definisi lama; mencakup serangan yang memang sudah lolos).
`base_recall` dilaporkan agar jelas seberapa sehat model sebelum diserang.

**Prasyarat:** `global_final.npz` dari run R3 ada di S3; partisi s01 ada.
Dilakukan **OFFLINE** (tak perlu EC2). **Output:** `survey_out/r5_evasion.csv`.
**MULTI-SEED.** Notebook ini membaca satu run federated per seed
(`SEEDS`, default `13,42,101,202,303`; seed 42 = run lama `S_R3`/`S_R6`, seed lain
`S_R3_s13`, ... ) dan melaporkan rata-rata, simpangan baku, dan setengah-lebar CI95
(distribusi t). Seed yang run-nya belum ada dilewati; kolom `n` menunjukkan jumlah seed
yang benar-benar dipakai. Variasi antar-seed mencakup inisialisasi bobot dan urutan
mini-batch; partisi data tetap (dibuat sekali di `s01`).


In [ ]:
import importlib, sys, subprocess
for m in ('numpy','scikit-learn','boto3'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, io, numpy as np
from sklearn.metrics import matthews_corrcoef
OUTDIR='survey_out'; os.makedirs(OUTDIR,exist_ok=True)
BUCKET='ssh-detection-features-232032302717'; REGION='ap-southeast-1'
BASE='unsw-far/federated_survey'
RUN=os.environ.get('RUN','S_R3')
import boto3; s3=boto3.client('s3',region_name=REGION)
def load_npz_s3(key):
    try: obj=s3.get_object(Bucket=BUCKET,Key=key)
    except Exception as e: print('MISS',key,e); return None
    return np.load(io.BytesIO(obj['Body'].read()),allow_pickle=True)
# --- MULTI-SEED: daftar seed + nama run per seed + rata-rata +/- CI95 (distribusi t) ---
import math
try:
    from scipy import stats as _st
except Exception:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scipy'], check=True)
    from scipy import stats as _st
SEEDS = [int(x) for x in os.environ.get('SEEDS', '13,42,101,202,303').split(',')]
def run_name(base, seed):
    # seed 42 = run lama (S_R3 / S_R6); seed lain = S_R3_s13, S_R6_s13, ...
    return base if seed == 42 else f'{base}_s{seed}'
def mean_ci(vals):
    v = np.array([x for x in vals if x == x], float)   # buang NaN
    n = len(v)
    if n == 0: return dict(n=0, mean=float('nan'), std=float('nan'), ci95=float('nan'))
    m = float(v.mean())
    if n == 1: return dict(n=1, mean=m, std=float('nan'), ci95=float('nan'))
    sd = float(v.std(ddof=1)); h = float(_st.t.ppf(0.975, n - 1) * sd / math.sqrt(n))
    return dict(n=n, mean=m, std=sd, ci95=h)
print('SEEDS =', SEEDS)

RUN_BASE = os.environ.get('RUN', 'S_R3')


## 1. Muat model global (R3) + data test; siapkan z-space & batas clip

In [ ]:
P = {}
for nm in ['cic_train', 'cic_test', 'unsw_train', 'unsw_test']:
    sub = 'cic' if nm.startswith('cic') else 'unsw'
    d = load_npz_s3(f'{BASE}/data/{sub}/{nm}.npz')
    P[nm] = (np.asarray(d['X'], float), np.asarray(d['y'], int))
CL = ['cic', 'unsw']
Z = {}   # per klien: data uji di z-space LOKAL (statistik dari TRAIN klien itu = sama dgn training FL)
for c in CL:
    Xtr_c = P[f'{c}_train'][0]
    m = Xtr_c.mean(0); s = Xtr_c.std(0); s[s == 0] = 1
    Ztr = (Xtr_c - m) / s
    Z[c] = dict(X=(P[f'{c}_test'][0] - m) / s, y=P[f'{c}_test'][1],
                lo=np.percentile(Ztr, 1, axis=0), hi=np.percentile(Ztr, 99, axis=0))
    print(c, 'test', Z[c]['X'].shape, 'pos', round(float(Z[c]['y'].mean()), 3))

## 2. Forward + gradien input (white-box)

In [ ]:
relu=lambda x:np.maximum(0,x); sig=lambda x:1/(1+np.exp(-np.clip(x,-30,30)))
def fwd(W,X):
    a1=X@W[0]+W[1]; z1=relu(a1); a2=z1@W[2]+W[3]; z2=relu(a2); a3=z2@W[4]+W[5]
    return sig(a3).ravel(),(a1,z1,a2,z2)
def dX_loss(W,X,y):
    o,(a1,z1,a2,z2)=fwd(W,X); g=(o-y.astype(float))[:,None]
    d2=(g@W[4].T)*(a2>0); d1=(d2@W[2].T)*(a1>0); return d1@W[0].T
def predict(W,X): o,_=fwd(W,X); return (o>=0.5).astype(int)
def mcc(W,X,y): return matthews_corrcoef(y,predict(W,X))

## 3. Serangan (semua CONSTRAINED: clip ke [lo,hi]). Target: flow ATTACK (y=1)

Evasion = buat serangan lolos sebagai benign. Kita ukur hanya pada sampel attack.

In [ ]:
def fgsm(W, X, y, eps, lo, hi):
    return np.clip(X + eps * np.sign(dX_loss(W, X, y)), lo, hi)

def pgd(W, X, y, eps, lo, hi, steps=10, alpha=None):
    alpha = alpha or eps / 4; Xa = X.copy()
    for _ in range(steps):
        Xa = Xa + alpha * np.sign(dX_loss(W, Xa, y))
        Xa = np.clip(np.clip(Xa, X - eps, X + eps), lo, hi)  # dalam bola-eps DAN batas clip
    return Xa

def jsma(W, X, y, eps, lo, hi, k=2):
    # saliency: pilih k fitur dgn |dL/dX| terbesar per-sampel, perturbasi hanya itu
    d = dX_loss(W, X, y); Xa = X.copy()
    idxk = np.argsort(-np.abs(d), axis=1)[:, :k]
    for i in range(len(X)):
        j = idxk[i]; Xa[i, j] = np.clip(X[i, j] + eps * np.sign(d[i, j]), lo[j], hi[j])
    return Xa

## 4. Sweep epsilon untuk tiap serangan -> attack success rate + ΔMCC

In [ ]:
def run_attack(W, fn, eps, **kw):
    out = {}; ys = []; p0s = []; p1s = []; dets = []
    for c in CL:
        X, y, lo, hi = Z[c]['X'], Z[c]['y'], Z[c]['lo'], Z[c]['hi']
        atk = (y == 1)
        p0 = predict(W, X)
        Xa = X.copy(); Xa[atk] = fn(W, X[atk], y[atk], eps, lo, hi, **kw)
        p1 = predict(W, Xa)
        det = atk & (p0 == 1)                      # serangan yang TERDETEKSI sebelum diserang
        out[c] = (y, p0, p1, det)
        ys.append(y); p0s.append(p0); p1s.append(p1); dets.append(det)
    out['union'] = (np.concatenate(ys), np.concatenate(p0s), np.concatenate(p1s), np.concatenate(dets))
    return out

def summarize(y, p0, p1, det):
    atk = (y == 1)
    return dict(base_recall=float((p0[atk] == 1).mean()),
                raw_miss=float((p1[atk] == 0).mean()),
                asr=float((p1[det] == 0).mean()) if det.sum() > 0 else float('nan'),
                mcc_clean=float(matthews_corrcoef(y, p0)),
                mcc_under=float(matthews_corrcoef(y, p1)))

rows = []
for seed in SEEDS:
    run = run_name(RUN_BASE, seed)
    gf = load_npz_s3(f'{BASE}/run_{run}/global_final.npz')
    if gf is None:
        print(f'[lewati] seed {seed}: run {run} belum ada di S3'); continue
    W = [gf[f'w{i}'] for i in range(6)]
    for eps in [0.05, 0.1, 0.2, 0.5]:
        for nm, fn in [('FGSM', fgsm), ('PGD', pgd), ('JSMA', jsma)]:
            res = run_attack(W, fn, eps)
            for scope, (y, p0, p1, det) in res.items():
                rows.append(dict(seed=seed, scope=scope, attack=nm, eps=eps, **summarize(y, p0, p1, det)))
            r = rows[-1]   # union
            print(f'seed {seed:<3} {nm:5s} eps={eps:<4} [union] base_recall={r["base_recall"]:.3f} '
                  f'ASR={r["asr"]:.3f} MCC {r["mcc_clean"]:+.3f} -> {r["mcc_under"]:+.3f}')
print('seed yang dipakai:', sorted({r['seed'] for r in rows}))

## 5. Simpan R5 -> CSV + S3

In [ ]:
import csv
f1 = ['seed', 'scope', 'attack', 'eps', 'base_recall', 'raw_miss', 'asr', 'mcc_clean', 'mcc_under']
p1 = os.path.join(OUTDIR, 'r5_evasion_seeds.csv')
with open(p1, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=f1); w.writeheader()
    for r in rows: w.writerow(r)
print('saved', p1)

keys = []
for r in rows:
    k = (r['scope'], r['attack'], r['eps'])
    if k not in keys: keys.append(k)
p2 = os.path.join(OUTDIR, 'r5_evasion_agg.csv')
metrics = ['base_recall', 'raw_miss', 'asr', 'mcc_clean', 'mcc_under']
with open(p2, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['scope', 'attack', 'eps', 'n'] + [f'{m}_{s}' for m in metrics for s in ('mean', 'std', 'ci95')])
    for k in keys:
        sel = [r for r in rows if (r['scope'], r['attack'], r['eps']) == k]
        stats_ = {m: mean_ci([r[m] for r in sel]) for m in metrics}
        w.writerow([*k, stats_['asr']['n']] + [stats_[m][s] for m in metrics for s in ('mean', 'std', 'ci95')])
print('saved', p2)
print('\n=== scope=union: ASR dan MCC-under, rata-rata +/- CI95 ===')
for k in keys:
    if k[0] == 'union':
        sel = [r for r in rows if (r['scope'], r['attack'], r['eps']) == k]
        a = mean_ci([r['asr'] for r in sel]); m = mean_ci([r['mcc_under'] for r in sel])
        print(f'{k[1]:5s} eps={k[2]:<4} ASR {a["mean"]:.3f}+/-{a["ci95"]:.3f}  MCC_under {m["mean"]:+.3f}+/-{m["ci95"]:.3f} (n={a["n"]})')
for pth, key in ((p1, 'r5_evasion_seeds.csv'), (p2, 'r5_evasion_agg.csv')):
    try:
        s3.upload_file(pth, BUCKET, f'{BASE}/results/{key}'); print('upload OK', key)
    except Exception as e:
        print('(upload dilewati:', e, ')')
print('=== s14 SELESAI. ===')